**TinyML Anomaly Detection using DCASE 2020 Slider dataset(subset of MIMII dataset)**

In [ ]:
import os
import glob
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import zipfile

# Install TensorFlow Model Optimization Toolkit
!pip install -q tensorflow-model-optimization
import tensorflow_model_optimization as tfmot

# ==========================================
# 1. DATASET SETUP (SLIDER)
# ==========================================
# Link for DCASE 2020 Slider (Dev Data)
dataset_url = "https://zenodo.org/record/3678171/files/dev_data_slider.zip?download=1"
zip_path = "dev_data_slider.zip"
extract_path = "./slider_dataset"

if not os.path.exists(zip_path):
    print(f"Downloading Slider Dataset... (Server speed)")
    !wget -O {zip_path} {dataset_url}

    print("Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print("Dataset already exists.")

# ==========================================
# 2. FEATURE ENGINEERING (Texture Focus)
# ==========================================
SAMPLE_RATE = 16000
n_mels = 64
n_frames = 64
n_fft = 1024
hop_length = 512

def file_to_vector_array(file_name):
    y, sr = librosa.load(file_name, sr=SAMPLE_RATE, mono=True)

    # Log-Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)

    # Sliding window (Strided)
    vectors = []
    # Stride of 8 makes training faster while keeping enough data
    for i in range(0, log_mel.shape[1] - n_frames, 8):
        vector = log_mel[:, i:i + n_frames].flatten()
        vectors.append(vector)
    return np.array(vectors)

# Load Files (MIMII/DCASE structure)
# slider/train (Normal)
# slider/test (Normal + Anomaly)
target_machine = "slider"
base_dir = f"{extract_path}/{target_machine}"

train_files = sorted(glob.glob(f"{base_dir}/train/*.wav"))
test_files = sorted(glob.glob(f"{base_dir}/test/*.wav"))

print(f"Found {len(train_files)} Train files, {len(test_files)} Test files")

# Process Train
print("Processing Train Data (Normal)...")
X_train_list = []
for fn in train_files:
    X_train_list.append(file_to_vector_array(fn))
X_train = np.concatenate(X_train_list)

# Process Test
print("Processing Test Data (Mixed)...")
X_test_list = []
y_test_list = []

for fn in test_files:
    vecs = file_to_vector_array(fn)
    X_test_list.append(vecs)

    # Label: Anomaly=1, Normal=0
    label = 1 if "anomaly" in os.path.basename(fn) else 0
    y_test_list.extend([label] * len(vecs))

X_test = np.concatenate(X_test_list)
y_test = np.array(y_test_list)

# Shuffle Test Data (Prevents NaN)
X_test, y_test = shuffle(X_test, y_test, random_state=42)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Feature Shape: {X_train.shape}") # Should be (N, 64*64 = 4096)

# ==========================================
# 3. BASELINE MODEL
# ==========================================
INPUT_DIM = n_mels * n_frames

model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(INPUT_DIM,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'), # Bottleneck
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(INPUT_DIM)
])

model.compile(optimizer='adam', loss='mse')

print("\n--- Training Baseline ---")
# Train on Normal data only
history = model.fit(
    X_train, X_train,
    epochs=15,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

# Eval Baseline
def get_auc(model, X, y):
    preds = model.predict(X, verbose=0)
    mse = np.mean(np.power(X - preds, 2), axis=1)
    return roc_auc_score(y, mse)

base_auc = get_auc(model, X_test, y_test)
print(f"Baseline AUC: {base_auc:.4f}")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible

In [ ]:
# ==========================================
# 4. PTQ & QAT
# ==========================================
def representative_data_gen():
    # Use first 100 samples for calibration
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

# --- PTQ ---
print("\nRunning PTQ...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_ptq = converter.convert()

with open('model_ptq.tflite', 'wb') as f: f.write(tflite_ptq)

# --- QAT ---
print("Running QAT (Retraining)...")
quant_model = tfmot.quantization.keras.quantize_model(model)
quant_model.compile(optimizer='adam', loss='mse')
quant_model.fit(X_train, X_train, batch_size=512, epochs=15, verbose=1)

converter_qat = tf.lite.TFLiteConverter.from_keras_model(quant_model)
converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
converter_qat.representative_dataset = representative_data_gen
converter_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_qat.inference_input_type = tf.int8
converter_qat.inference_output_type = tf.int8
tflite_qat = converter_qat.convert()

with open('model_qat.tflite', 'wb') as f: f.write(tflite_qat)


Running PTQ...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Running QAT (Retraining)...
Epoch 1/15
176/176 [==============================] - 7s 16ms/step - loss: 0.3120
Epoch 2/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2508
Epoch 3/15
176/176 [==============================] - 3s 19ms/step - loss: 0.2412
Epoch 4/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2372
Epoch 5/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2354
Epoch 6/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2334
Epoch 7/15
176/176 [==============================] - 3s 17ms/step - loss: 0.2326
Epoch 8/15
176/176 [==============================] - 3s 16ms/step - loss: 0.2311
Epoch 9/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2307
Epoch 10/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2303
Epoch 11/15
176/176 [==============================] - 3s 15ms/step - loss: 0.2296
Epoch 12/15
176/176 [==============================] - 3s 19ms/step 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
# ==========================================
# 5. FINAL COMPARISON & TABLE GENERATION
# ==========================================
import time
import pandas as pd

# 1. Save Baseline as Float32 TFLite (for fair size/latency comparison)
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter_float.convert()
with open('model_float32.tflite', 'wb') as f:
    f.write(tflite_float)

# Helper: Measure Latency (Inference Time on CPU)
def get_latency_ms(tflite_path, X_sample):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Pre-process input to match model type
    scale, zero_point = input_details[0]['quantization']
    if scale > 0: # Integer model
        input_data = X_sample / scale + zero_point
        input_data = input_data.astype(np.int8)
    else: # Float model
        input_data = X_sample.astype(np.float32)

    input_data = np.expand_dims(input_data, axis=0)

    # Warmup
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    # Measure
    start_time = time.time()
    for _ in range(100): # Run 100 times
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
    end_time = time.time()

    return ((end_time - start_time) / 100) * 1000 # Convert to ms

# Helper: Evaluate AUC for TFLite
def eval_tflite_auc(tflite_path, X, y):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    scale_in, zero_in = input_details[0]['quantization']
    scale_out, zero_out = output_details[0]['quantization']

    # Use subset for speed
    limit = 3000
    X_sub, y_sub = X[:limit], y[:limit]
    mses = []

    for i in range(len(X_sub)):
        # Quantize Input if needed
        if scale_in > 0:
            in_data = X_sub[i] / scale_in + zero_in
            in_data = np.expand_dims(in_data.astype(np.int8), axis=0)
        else:
            in_data = np.expand_dims(X_sub[i].astype(np.float32), axis=0)

        interpreter.set_tensor(input_details[0]['index'], in_data)
        interpreter.invoke()
        out_data = interpreter.get_tensor(output_details[0]['index'])

        # Dequantize Output if needed
        if scale_out > 0:
            out_data = (out_data.astype(np.float32) - zero_out) * scale_out

        mses.append(np.mean((X_sub[i] - out_data.flatten())**2))

    return roc_auc_score(y_sub, mses)

def get_size_kb(path):
    return os.path.getsize(path) / 1024

print("\nRunning Final Evaluation (This may take a minute)...")

# --- Gather Metrics ---
# 1. Baseline (Float32)
auc_base = eval_tflite_auc('model_float32.tflite', X_test, y_test)
lat_base = get_latency_ms('model_float32.tflite', X_test[0])
size_base = get_size_kb('model_float32.tflite')

# 2. PTQ (Int8)
auc_ptq = eval_tflite_auc('model_ptq.tflite', X_test, y_test)
lat_ptq = get_latency_ms('model_ptq.tflite', X_test[0])
size_ptq = get_size_kb('model_ptq.tflite')

# 3. QAT (Int8)
auc_qat = eval_tflite_auc('model_qat.tflite', X_test, y_test)
lat_qat = get_latency_ms('model_qat.tflite', X_test[0])
size_qat = get_size_kb('model_qat.tflite')

# --- Generate Table ---
results_data = {
    "Model Type": ["Baseline (Float32)", "PTQ (Int8)", "QAT (Int8)"],
    "Size (KB)": [f"{size_base:.2f}", f"{size_ptq:.2f}", f"{size_qat:.2f}"],
    "Accuracy (AUC)": [f"{auc_base:.4f}", f"{auc_ptq:.4f}", f"{auc_qat:.4f}"],
    "Latency (ms)*": [f"{lat_base:.3f}", f"{lat_ptq:.3f}", f"{lat_qat:.3f}"]
}

df = pd.DataFrame(results_data)

print("\n" + "="*60)
print("FINAL PERFORMANCE COMPARISON (Slider Dataset)")
print("="*60)
print(df.to_string(index=False))
print("-" * 60)
print("* Latency measured as average inference time on Colab CPU.")
print("  Real microcontroller latency will be different but follows similar ratios.")
print("="*60)


Running Final Evaluation (This may take a minute)...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 


FINAL PERFORMANCE COMPARISON (Slider Dataset)
        Model Type Size (KB) Accuracy (AUC) Latency (ms)*
Baseline (Float32)   4277.10         0.8011         0.190
        PTQ (Int8)   1194.32         0.7982         0.059
        QAT (Int8)   1086.29         0.8169         0.065
------------------------------------------------------------
* Latency measured as average inference time on Colab CPU.
  Real microcontroller latency will be different but follows similar ratios.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


The table presents a comparison of three model types:

*   **Baseline (Float32)**: This is the original model, trained with standard 32-bit floating-point precision.
*   **PTQ (Int8)**: This model underwent Post-Training Quantization. It was converted to 8-bit integer precision *after* training, without any further retraining.
*   **QAT (Int8)**: This model underwent Quantization-Aware Training. It was trained with quantization simulated during the training process, resulting in a model that is better optimized for 8-bit integer precision.

Let's break down the metrics:

1.  **Size (KB)**: This metric indicates the size of the model file in kilobytes. Smaller is better, especially for edge devices with limited memory.
    *   The **Baseline** model is the largest at 4277.10 KB.
    *   **PTQ** significantly reduces the size to 1194.32 KB (a reduction of about 72%).
    *   **QAT** further reduces the size slightly to 1086.29 KB (about 75% reduction from baseline).

2.  **Accuracy (AUC)**: AUC (Area Under the Receiver Operating Characteristic Curve) is a common metric for anomaly detection models. A higher AUC indicates better performance, with 1.0 being perfect and 0.5 being random chance.
    *   The **Baseline** achieved an AUC of 0.8011.
    *   **PTQ** shows a very slight drop to 0.7982, which is good as it maintains most of the original accuracy.
    *   **QAT** demonstrates an improved AUC of 0.8169, surpassing both the Baseline and PTQ models. This indicates that the retraining process during Quantization-Aware Training successfully optimized the model for 8-bit integer precision, leading to better performance.

3.  **Latency (ms)***: This measures the average time it takes for the model to perform a single inference on the Colab CPU, in milliseconds. Lower latency means faster predictions, which is crucial for real-time applications and low-power devices.
    *   The **Baseline** model has a latency of 0.190 ms.
    *   **PTQ** drastically reduces latency to 0.059 ms (a reduction of about 69%).
    *   **QAT** achieves a similar low latency of 0.065 ms.

In summary, these results show that quantization, especially QAT, can lead to substantial improvements in model efficiency (size and speed) while maintaining or even enhancing accuracy, making it highly suitable for TinyML anomaly detection applications.